In [1]:
from multiprocessing import Process, Manager
from threading import Thread
import datetime
import time
import pandas as pd
import websocket
import json

from binance.client import Client # Temporary
from binance import ThreadedWebsocketManager # Works fine only with python-binanc==1.0.12 Version
from upbit.client import Upbit

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

[WARNING] upbit-client is currently a newer version: 1.3.2.0
Please update to the latest version using the pip command:`pip install --upgrade upbit-client`


In [2]:
def fetch_dollar(url='https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=1&ie=utf8&query=%ED%99%98%EC%9C%A8%EC%A1%B0%ED%9A%8C'):
    # global DOLLAR_INFO_DICT
    DOLLAR_INFO_DICT = None
    try:
        exchange_rate = pd.read_html(url)[0]
        DOLLAR_INFO_DICT['price'] = exchange_rate.iloc[0,1]
        DOLLAR_INFO_DICT['change'] = exchange_rate.iloc[0,-1]
        DOLLAR_INFO_DICT['last_updated_time'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    except Exception as e:
        print(f'Except executed in get_dollar function, {e}')
        exchange_rate = pd.read_html(url)[1]
        DOLLAR = exchange_rate.iloc[0,1]
        DOLLAR_INFO_DICT['price'] = exchange_rate.iloc[0,1]
        DOLLAR_INFO_DICT['change'] = exchange_rate.iloc[0,-1]
        DOLLAR_INFO_DICT['last_updated_time'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return DOLLAR_INFO_DICT


In [3]:
url='https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=1&ie=utf8&query=%ED%99%98%EC%9C%A8%EC%A1%B0%ED%9A%8C'
pd.read_html(url)[0].iloc[0,1]

1383.0

In [4]:
datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

'2022-09-12 13:57:03'

# Extract both listed symbols

In [5]:
def extract_shared_listed_symbols():
    bi_client = Client()
    upbit_client = Upbit()
    binance_ticker_df = pd.DataFrame(bi_client.futures_ticker())
    binance_ticker_df.loc[:, ['lastPrice','quoteVolume']] = binance_ticker_df[['lastPrice','quoteVolume']].astype(float)
    usdm_listed_symbols = binance_ticker_df[(binance_ticker_df['symbol'].str.contains('USDT'))&(~binance_ticker_df['symbol'].str.contains('_'))]['symbol'].to_list()
    # # Upbit listed_symbols
    upbit_symbols_df = pd.DataFrame(upbit_client.Market.Market_info_all()['result'])
    upbit_symbols = upbit_symbols_df[upbit_symbols_df['market'].str.contains('KRW')]['market'].to_list()
    upbit_symbols = [x.replace('KRW-','')+'USDT' for x in upbit_symbols]
    # Binance USD-M & Upbit
    both_listed_binance_symbols = list(set(usdm_listed_symbols).intersection(set(upbit_symbols)))
    # Reflecting trading KRW volumes, decending order
    both_listed_binance_symbols = binance_ticker_df[binance_ticker_df['symbol'].isin(both_listed_binance_symbols)][['symbol','quoteVolume']]\
        .sort_values('quoteVolume', ascending=False)['symbol'].to_list()
    both_listed_upbit_symbols = ['KRW-' + x.replace('USDT','') for x in both_listed_binance_symbols]
    # Reflecting trading KRW volumes, decending order
    both_listed_upbit_symbols = pd.DataFrame(upbit_client.Trade.Trade_ticker(markets=','.join(both_listed_upbit_symbols))\
        ['result'])[['market','acc_trade_price_24h']].sort_values('acc_trade_price_24h', ascending=False)['market'].to_list()
    return {'both_listed_binance_symbols': both_listed_binance_symbols, 'both_listed_upbit_symbols': both_listed_upbit_symbols}

def list_slice(lst, n):
    sliced_nested_list = []
    for i in range(n):
        sliced_nested_list.append(lst[i::n])
    return sliced_nested_list

# Binance Websocket

In [11]:
def binance_bookticker_websocket(both_listed_binance_symbols, proc_n):
    global BINANCE_BOOKTICKER_DICT
    global BINANCE_BOOKTICKER_PROC_LIST

    def binance_websocket(BINANCE_BOOKTICKER_DICT, both_listed_binance_symbols):
        def handle_ticker_streams(msg):
            # event_type = msg['stream']
            # print(msg['data']) # test
            # if msg['data']['s'] == 'ETHUSDT': # test
            #     print(msg['data']['T']) # test
            BINANCE_BOOKTICKER_DICT[msg['data']['s']] = msg['data']
        binance_websocket_thread = ThreadedWebsocketManager()
        binance_websocket_thread.daemon = True
        binance_websocket_thread.start()
        ticker_list = [x.lower()+'@bookTicker' for x in both_listed_binance_symbols]
        binance_websocket_thread.start_futures_multiplex_socket(callback=handle_ticker_streams, streams=ticker_list)
        binance_websocket_thread.join()

    manager = Manager()
    BINANCE_BOOKTICKER_DICT = manager.dict()
    BINANCE_BOOKTICKER_PROC_LIST = []
    sliced_both_listed_binance_symbols_list = list_slice(both_listed_binance_symbols, proc_n)

    for i, each_both_listed_binance_symbols in enumerate(sliced_both_listed_binance_symbols_list):
        binance_bookticker_proc = Process(target=binance_websocket, args=(BINANCE_BOOKTICKER_DICT, each_both_listed_binance_symbols), daemon=True)
        BINANCE_BOOKTICKER_PROC_LIST.append(binance_bookticker_proc)
        binance_bookticker_proc.start()
        print(f"{i+1}th binance_bookticker_proc started.")
        time.sleep(0.3)

In [7]:
# binance_bookticer_websocket(both_listed_binance_symbols, 4)

In [8]:
# # Terminate
# for i, each_proc in enumerate(binance_bookticker_proc_list):
#     print(f"Terminating {i+1}th proc..")
#     each_proc.terminate()

# time.sleep(1)

# for i, each_proc in enumerate(binance_bookticker_proc_list):
#     print(f"{i+1}th proc is_alive status: {each_proc.is_alive()}")

# Upbit Websocket

In [12]:
def upbit_orderbook_ticker_websocket(both_listed_upbit_symbols, proc_n):
    global UPBIT_TICKER_DICT
    global UPBIT_ORDERBOOK_DICT
    global UPBIT_TICKER_PROC_LIST
    global UPBIT_ORDERBOOK_PROC_LIST

    def upbit_websocket(upbit_result_dict, data):
        def on_message(ws, message):
            message_dict = json.loads(message)
            # if message_dict['cd'] == 'KRW-BCH':                                                         # test
            #     print(message_dict['tp'], datetime.datetime.fromtimestamp(message_dict['tms']/1000))    # test
            upbit_result_dict[message_dict['cd']] = message_dict

        def on_error(ws, error):
            print(f'on_error executed!')
            print(error)
            pass

        def on_close(ws, close_status_code, close_msg):
            print(f"\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")

        def on_open(ws):
            print(f'upbit_websocket started')
            ws.send(json.dumps(data))

        websocket.enableTrace(False)
        ws = websocket.WebSocketApp("wss://api.upbit.com/websocket/v1",
                                on_open=on_open,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close)
        ws.run_forever()

    sliced_both_listed_upbit_symbols_list = list_slice(both_listed_upbit_symbols, proc_n)

    manager = Manager()
    UPBIT_TICKER_DICT = manager.dict()
    UPBIT_ORDERBOOK_DICT = manager.dict()

    UPBIT_TICKER_PROC_LIST = []
    UPBIT_ORDERBOOK_PROC_LIST = []

    for i, each_both_listed_upbit_symbols in enumerate(sliced_both_listed_upbit_symbols_list):
        upbit_ticker_data = [{"ticket":"people9"},{"type":"ticker","codes":each_both_listed_upbit_symbols}, {"format":"SIMPLE"}]
        upbit_orderbook_data = [{"ticket":"people9"},{"type":"orderbook","codes":[x+'.1' for x in each_both_listed_upbit_symbols]}, {"format":"SIMPLE"}]

        upbit_ticker_proc = Process(target=upbit_websocket, args=(UPBIT_TICKER_DICT, upbit_ticker_data), daemon=True)
        UPBIT_TICKER_PROC_LIST.append(upbit_ticker_proc)
        upbit_ticker_proc.start()
        print(f"{i+1}th upbit_ticker_proc started.")
        time.sleep(0.3)

        upbit_orderbook_proc = Process(target=upbit_websocket, args=(UPBIT_ORDERBOOK_DICT, upbit_orderbook_data), daemon=True)
        UPBIT_ORDERBOOK_PROC_LIST.append(upbit_orderbook_proc)
        upbit_orderbook_proc.start()
        print(f"{i+1}th upbit_orderbook_proc started.")
        time.sleep(0.3)

In [13]:
# upbit_orderbook_ticker_websocket(both_listed_upbit_symbols, 3)

# Both websockets

In [14]:
def binance_upbit_websocket(both_listed_binance_symbols, both_listed_upbit_symbols, proc_n):
    binance_bookticker_websocket_th = Thread(target=binance_bookticker_websocket, args=(both_listed_binance_symbols, proc_n), daemon=True)
    binance_bookticker_websocket_th.start()
    upbit_orderbook_ticker_websocket(both_listed_upbit_symbols, proc_n)
    binance_bookticker_websocket_th.join()

def terminate_websocket_proc(BINANCE_BOOKTICKER_PROC_LIST=None, UPBIT_TICKER_PROC_LIST=None, UPBIT_ORDERBOOK_PROC_LIST=None):
    if BINANCE_BOOKTICKER_PROC_LIST is not None:
        for i, each_proc in enumerate(BINANCE_BOOKTICKER_PROC_LIST):
            print(f"terminating {i+1}th binance_bookticker_proc..")
            each_proc.terminate()
    if UPBIT_TICKER_PROC_LIST is not None:
        for i, each_proc in enumerate(UPBIT_TICKER_PROC_LIST):
            print(f"terminating {i+1}th upbit_ticker_proc..")
            each_proc.terminate()
    if UPBIT_ORDERBOOK_PROC_LIST is not None:
        for i, each_proc in enumerate(UPBIT_ORDERBOOK_PROC_LIST):
            print(f"terminating {i+1}th upbit_orderbook_proc..")
            each_proc.terminate()
    time.sleep(2)
    if BINANCE_BOOKTICKER_PROC_LIST is not None:
        for i, each_proc in enumerate(BINANCE_BOOKTICKER_PROC_LIST):
            print(f"{i+1}th binance_bookticker_proc is_alive: {each_proc.is_alive()}")
    if UPBIT_TICKER_PROC_LIST is not None:
        for i, each_proc in enumerate(UPBIT_TICKER_PROC_LIST):
            print(f"{i+1}th upbit_ticker_proc is_alive: {each_proc.is_alive()}")
    if UPBIT_ORDERBOOK_PROC_LIST is not None:
        for i, each_proc in enumerate(UPBIT_ORDERBOOK_PROC_LIST):
            print(f"{i+1}th upbit_orderbook_proc is_alive: {each_proc.is_alive()}")

In [15]:
shared_listed_symbol_dict = extract_shared_listed_symbols()
both_listed_binance_symbols = shared_listed_symbol_dict['both_listed_binance_symbols']
both_listed_upbit_symbols = shared_listed_symbol_dict['both_listed_upbit_symbols']

binance_upbit_websocket(both_listed_binance_symbols, both_listed_upbit_symbols, 1)

1th binance_bookticker_proc started.
upbit_websocket started
1th upbit_ticker_proc started.
upbit_websocket started
1th upbit_orderbook_proc started.


In [30]:
terminate_websocket_proc(BINANCE_BOOKTICKER_PROC_LIST=BINANCE_BOOKTICKER_PROC_LIST, UPBIT_TICKER_PROC_LIST=UPBIT_TICKER_PROC_LIST, UPBIT_ORDERBOOK_PROC_LIST=UPBIT_ORDERBOOK_PROC_LIST)

terminating 1th binance_bookticker_proc..
terminating 1th upbit_ticker_proc..
terminating 1th upbit_orderbook_proc..
1th binance_bookticker_proc is_alive: False
1th upbit_ticker_proc is_alive: False
1th upbit_orderbook_proc is_alive: False


# Converting methods

In [26]:
def binance_bookticker_convert(BINANCE_BOOKTICKER_DICT):
    BINANCE_BOOKTICKER_DICT_copy = dict(BINANCE_BOOKTICKER_DICT.copy())
    converted_df = pd.DataFrame(BINANCE_BOOKTICKER_DICT_copy).transpose().reset_index(drop=True)
    # binance_bookTicker_column_dict = {
    #     'e':'event_type',
    #     'u':'order_book_updateId',
    #     'E':'event_time',
    #     'T':'transaction_time',
    #     's':'symbol',
    #     'b':'best_bid_price',
    #     'B':'best_bid_qty',
    #     'a':'best_ask_price',
    #     'A':'best_ask_qty'
    # }
    # temp_df = temp_df.rename(columns=binance_bookTicker_column_dict)
    return converted_df

def upbit_ticker_convert(UPBIT_TICKER_DICT):
    UPBIT_TICKER_DICT_copy = dict(UPBIT_TICKER_DICT.copy())
    converted_df = pd.DataFrame(UPBIT_TICKER_DICT_copy).transpose().reset_index(drop=True)
    # upbit_column_dict = {
    #     'ty':'type',
    #     'cd':'code',
    #     'op':'opening_price',
    #     'hp':'high_price',
    #     'lp':'low_price',
    #     'tp':'trade_price',
    #     'pcp':'prev_closing_price',
    #     'c':'change',
    #     'cp':'change_price',
    #     'scp':'signed_change_price',
    #     'cr':'change_rate',
    #     'scr':'signed_change_rate',
    #     'tv':'trade_volume',
    #     'atv':'acc_trade_volume',
    #     'atv24h':'acc_trade_volume_24h',
    #     'atp':'acc_trade_price',
    #     'atp24h':'acc_trade_price_24h',
    #     'tdt':'trade_date',
    #     'ttm':'trade_time',
    #     'ttms':'trade_timestamp',
    #     'ab':'ask_bid',
    #     'aav':'acc_ask_volume',
    #     'abv':'acc_bid_volume',
    #     'h52wp':'highest_52_week_price',
    #     'h52wdt':'highest_52_week_date',
    #     'l52wp':'lowest_52_week_price',
    #     'l52wdt':'lowest_52_week_date',
    #     'ts':'trade_status',
    #     'ms':'market_state',
    #     'msfi':'market_state_for_ios',
    #     'its':'is_trading_suspended',
    #     'dd':'delisting_date',
    #     'mw':'market_warning',
    #     'tms':'timestamp',
    #     'st':'stream_type'
    # }
    # temp_df = temp_df.rename(columns=upbit_column_dict)
    return converted_df

def upbit_orderbook_convert(UPBIT_ORDERBOOK_DICT):
    UPBIT_ORDERBOOK_DICT_copy = dict(UPBIT_ORDERBOOK_DICT.copy())
    converted_df = pd.DataFrame(UPBIT_ORDERBOOK_DICT_copy).transpose().reset_index(drop=True)
    # upbit_column_dict = {
    #     'ty':'type',
    #     'cd':'code',
    #     'tas':'total_ask_size',
    #     'tbs':'total_bid_size',
    #     'obu':'orderbook_units',
    #     'tms':'timestamp',
    #     'st':'stream_type'
    # }
    # temp_df = temp_df.rename(columns=upbit_column_dict)
    return converted_df

def get_kimp_df(BINANCE_BOOKTICKER_DICT, UPBIT_TICKER_DICT, UPBIT_ORDERBOOK_DICT, current_dollar):
    upbit_allticker_df = upbit_ticker_convert(UPBIT_TICKER_DICT)
    binance_bookTickers = binance_bookticker_convert(BINANCE_BOOKTICKER_DICT) 
    upbit_orderbook_df = upbit_orderbook_convert(UPBIT_ORDERBOOK_DICT)
    upbit_allticker_df = upbit_allticker_df.merge(upbit_orderbook_df, on='cd')
    upbit_allticker_df.loc[:, 'cd'] = upbit_allticker_df['cd'].apply(lambda x: x.replace('KRW-', '') + 'USDT')
    merged_allticker_df = upbit_allticker_df.merge(binance_bookTickers, left_on='cd', right_on='s')
    merged_allticker_df.loc[:, 'b'] = merged_allticker_df['b'].astype(float)
    merged_allticker_df.loc[:, 'a'] = merged_allticker_df['a'].astype(float)
    # Overwrite binance last_price to binance best bid price
    # merged_allticker_df = merged_allticker_df.rename(columns={'best_bid_price': 'last_price'})
    merged_allticker_df['binance_bid_price_krw'] = merged_allticker_df['b'] * current_dollar
    merged_allticker_df['binance_ask_price_krw'] = merged_allticker_df['a'] * current_dollar
    # Overwrite upbit trade_price to upbit best ask price
    merged_allticker_df['upbit_ask_price'] = merged_allticker_df['obu'].apply(lambda x: x[0]['ap'])
    merged_allticker_df['upbit_bid_price'] = merged_allticker_df['obu'].apply(lambda x: x[0]['bp'])
    merged_allticker_df['enter_kimp'] = (merged_allticker_df['upbit_ask_price'] - merged_allticker_df['binance_bid_price_krw']) / merged_allticker_df['binance_bid_price_krw']
    merged_allticker_df['exit_kimp'] = (merged_allticker_df['upbit_bid_price'] - merged_allticker_df['binance_ask_price_krw']) / merged_allticker_df['binance_ask_price_krw']
    merged_allticker_df['tp_kimp'] = (merged_allticker_df['tp'] - merged_allticker_df['binance_bid_price_krw']) / merged_allticker_df['binance_bid_price_krw']
    merged_allticker_df['enter_usdt'] = (merged_allticker_df['enter_kimp'] + 1) * current_dollar
    merged_allticker_df['exit_usdt'] = (merged_allticker_df['exit_kimp'] + 1) * current_dollar
    merged_allticker_df['tp_usdt'] = (merged_allticker_df['tp_kimp'] + 1) * current_dollar
    merged_allticker_df['upbit_ticker_ratio'] = merged_allticker_df['tp'].apply(lambda x: get_ticker_ratio(x))
    merged_allticker_df = merged_allticker_df.rename(columns={
        's':'symbol',
        'atp24h':'acc_trade_price_24h',
        'tp':'trade_price',
        'b':'binance_bid_price',
        'a':'binance_ask_price',
        'scr':'signed_change_rate',
        'ms':'upbit_market_state',
        'dd':'upbit_delisting_date',
        'mw':'upbit_market_warning',
        'its':'upbit_is_trading_suspended',
        'tms_x':'upbit_timestamp',
        'E':'binance_event_time'
    })
    kimp_df = (merged_allticker_df[['symbol','acc_trade_price_24h','trade_price','upbit_ticker_ratio','upbit_ask_price','upbit_bid_price',
    'binance_bid_price','binance_bid_price_krw','binance_ask_price','binance_ask_price_krw','signed_change_rate','upbit_market_state','upbit_delisting_date',
    'upbit_market_warning','upbit_is_trading_suspended','upbit_timestamp','binance_event_time',
    'enter_kimp', 'exit_kimp', 'tp_kimp',
    'enter_usdt', 'exit_usdt', 'tp_usdt']])
    return kimp_df

In [28]:
len(binance_bookticker_convert(BINANCE_BOOKTICKER_DICT))

53

In [29]:
kimp_df = get_kimp_df(BINANCE_BOOKTICKER_DICT, UPBIT_TICKER_DICT, UPBIT_ORDERBOOK_DICT, 1340)
kimp_df

,symbol,acc_trade_price_24h,trade_price,upbit_ticker_ratio,upbit_ask_price,upbit_bid_price,binance_bid_price,binance_bid_price_krw,binance_ask_price,binance_ask_price_krw,...,upbit_market_warning,upbit_is_trading_suspended,upbit_timestamp,binance_event_time,enter_kimp,exit_kimp,tp_kimp,enter_usdt,exit_usdt,tp_usdt
0,ETCUSDT,236194852024.071686,45820.0,0.000218,45820.0,45810.0,33.549000,4.495566e+04,33.55000,4.495700e+04,...,NONE,False,1661318852742,1661318852752,0.019227,0.018974,0.019227,1365.763510,1365.424739,1365.76351
1,CHZUSDT,184539627146.702911,338.0,0.002959,338.0,337.0,0.247190,3.312346e+02,0.24720,3.312480e+02,...,NONE,False,1661318850573,1661318852801,0.020425,0.017365,0.020425,1367.369230,1363.268608,1367.36923
2,EOSUSDT,170515467839.438446,2395.0,0.002088,2400.0,2395.0,1.754000,2.350360e+03,1.75500,2.351700e+03,...,NONE,False,1661318850238,1661318852752,0.021120,0.018412,0.018993,1368.301026,1364.672365,1365.450399
3,ETHUSDT,160782875488.092163,2230000.0,0.000448,2231000.0,2230000.0,1634.540000,2.190284e+06,1634.55000,2.190297e+06,...,NONE,False,1661318850059,1661318852791,0.018590,0.018127,0.018133,1364.910005,1364.289866,1364.298212
4,WAVESUSDT,106270471237.642609,6890.0,0.000726,6890.0,6885.0,5.039000,6.752260e+03,5.04000,6.753600e+03,...,NONE,False,1661318851960,1661318852800,0.020399,0.019456,0.020399,1367.334789,1366.071429,1367.334789
5,BTCUSDT,105975066696.613388,29206000.0,0.000034,29206000.0,29191000.0,21404.800000,2.868243e+07,21404.90000,2.868257e+07,...,NONE,False,1661318850054,1661318852729,0.018254,0.017726,0.018254,1364.460308,1363.753159,1364.460308
6,XRPUSDT,78452966295.005081,467.0,0.002141,468.0,467.0,0.342300,4.586820e+02,0.34240,4.588160e+02,...,NONE,False,1661318850124,1661318852798,0.020315,0.017837,0.018135,1367.221735,1363.901869,1364.300321
7,BCHUSDT,67233890744.174698,180550.0,0.000277,180800.0,180550.0,132.480000,1.775232e+05,132.49000,1.775366e+05,...,NONE,False,1661318790300,1661318852752,0.018458,0.016973,0.01705,1364.734300,1362.744358,1362.847222
8,ATOMUSDT,66333324946.065269,16240.0,0.000616,16240.0,16230.0,11.888000,1.592992e+04,11.88900,1.593126e+04,...,NONE,False,1661318850485,1661318852741,0.019465,0.018752,0.019465,1366.083445,1365.127429,1366.083445
9,SOLUSDT,42713698715.843002,47730.0,0.000210,47740.0,47730.0,34.960000,4.684640e+04,34.97000,4.685980e+04,...,NONE,False,1661318852167,1661318852752,0.019075,0.018570,0.018862,1365.560641,1364.884186,1365.2746
